In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Audio, display

import librosa
import librosa.display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from huggingface_hub import snapshot_download, login as hf_login
from datasets import load_dataset

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")

In [ ]:
ROOT_PATH = Path("../").resolve()
DATA_PATH = ROOT_PATH / "data"

## Dataset

In [ ]:
repo_id = "doof-ferb/fpt_fosd"
local_dir = "../data/fpt_fosd"

downloaded_path = snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    local_dir=local_dir,
)

In [ ]:
DATASET_PATH = DATA_PATH / "fpt_fosd" / "data"
dataset = load_dataset("parquet", data_dir=DATASET_PATH)
dataset

In [ ]:
sample = dataset["train"][0]
sample

In [ ]:
audio_data = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]
print("Audio shape:", audio_data.shape)
print("Sampling rate:", sampling_rate)

In [ ]:
Audio(data=audio_data, rate=sampling_rate)

In [ ]:
# Create an array of time values (in seconds) for the x-axis
time_axis = np.linspace(0, len(audio_data) / sampling_rate, num=len(audio_data))
plt.figure(figsize=(10, 3))
plt.plot(time_axis, audio_data, color="blue")
plt.title("Audio Waveform")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.xlim(
    [0, time_axis[-1]]
)  # Ensure the x-axis starts at 0 and ends at the exact length
plt.tight_layout()
plt.show()

In [ ]:
# Create an array of time values (in seconds) for the x-axis
start_time = 2_500
end_time = 3_500
time_axis = np.linspace(
    0,
    len(audio_data[start_time:end_time]) / sampling_rate,
    num=len(audio_data[start_time:end_time]),
)
plt.figure(figsize=(10, 3))
plt.plot(time_axis, audio_data[start_time:end_time], color="cyan")
plt.title("Audio Waveform")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.xlim(
    [0, time_axis[-1]]
)  # Ensure the x-axis starts at 0 and ends at the exact length
plt.tight_layout()
plt.show()

In [ ]:
# 1. Compute the Mel Spectrogram
mel_spectrogram = librosa.feature.melspectrogram(
    y=audio_data,
    sr=sampling_rate,
    n_mels=128,  # Number of Mel bands to generate
    n_fft=2048,  # Length of the FFT window
    hop_length=512,  # Number of samples between successive frames
    fmax=8000,  # Maximum frequency (optional, often half the SR is good)
)

In [ ]:
mel_spectrogram

In [ ]:
log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max)
print("Log-Mel shape:", log_mel_spectrogram.shape)
# ---------------------------------------------------------
# 3. (Optional) Visualize the Log-Mel Spectrogram
# ---------------------------------------------------------
plt.figure(figsize=(10, 4))
librosa.display.specshow(
    log_mel_spectrogram,
    sr=sampling_rate,
    hop_length=512,
    x_axis="time",
    y_axis="mel",
    fmax=8000,
)
plt.colorbar(format="%+2.0f dB")
plt.title("Log-Mel Spectrogram")
plt.tight_layout()
plt.show()

In [ ]:
# ── 1. Load audio ──────────────────────────────────────────────
# y, sr = librosa.load("audio.wav", sr=16000)  # resample to 16kHz
y = audio_data
sr = sampling_rate
# y: float32 array, normalized to [-1.0, 1.0]

# ── 2. Pre-emphasis ────────────────────────────────────────────
y_emp = np.append(y[0], y[1:] - 0.97 * y[:-1])

# ── 3. STFT → Power Spectrogram ────────────────────────────────
n_fft = 512  # ~32ms window at 16kHz
hop_len = 128  # ~8ms hop → 75% overlap
window = "hann"

D = librosa.stft(y_emp, n_fft=n_fft, hop_length=hop_len, window=window)
S_power = np.abs(D) ** 2  # shape: (257, T)

# ── 4. Mel Filterbank ──────────────────────────────────────────
mel_fb = librosa.filters.mel(
    sr=sr, n_fft=n_fft, n_mels=80, fmin=0, fmax=8000
)  # shape: (80, 257)
S_mel = mel_fb @ S_power  # shape: (80, T)

# ── 5. Log compression ─────────────────────────────────────────
S_log_mel = np.log(S_mel + 1e-9)  # shape: (80, T)

# ── 6. MFCC (optional, classical) ─────────────────────────────
mfccs = librosa.feature.mfcc(S=S_log_mel, n_mfcc=13)
delta = librosa.feature.delta(mfccs)
delta2 = librosa.feature.delta(mfccs, order=2)
mfcc_full = np.vstack([mfccs, delta, delta2])  # shape: (39, T)

# ── 7. One-liner equivalents (librosa shorthand) ───────────────
S_log_mel_fast = librosa.power_to_db(
    librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_len, n_mels=80
    ),
    ref=np.max,
)

# ── 8. Visualize ───────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(10, 8))

librosa.display.waveshow(y, sr=sr, ax=axes[0])
axes[0].set_title("Waveform")

librosa.display.specshow(
    librosa.amplitude_to_db(np.abs(D)),
    sr=sr,
    hop_length=hop_len,
    x_axis="time",
    y_axis="hz",
    ax=axes[1],
)
axes[1].set_title("Spectrogram (linear freq)")

librosa.display.specshow(
    S_log_mel, sr=sr, hop_length=hop_len, x_axis="time", y_axis="mel", ax=axes[2]
)
axes[2].set_title("Log-Mel Spectrogram")

plt.tight_layout()
# plt.savefig("pipeline.png", dpi=150)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ══════════════════════════════════════════════════════════════
#  PLUG YOUR DATA IN HERE
# ══════════════════════════════════════════════════════════════
SR: int = sampling_rate
wave: list = audio_data

# If you have no wave yet, we generate a demo one
if wave is None:
    t = np.linspace(0, 1.0, SR, endpoint=False)
    wave = (np.sin(2 * np.pi * 300 * t) + 0.5 * np.sin(2 * np.pi * 3000 * t)).tolist()
    print("[demo mode] using synthetic 300 Hz + 3000 Hz wave")

# ══════════════════════════════════════════════════════════════
#  PIPELINE PARAMS
# ══════════════════════════════════════════════════════════════
N_FFT = 512  # samples per window  (~32ms at 16kHz)
HOP_LEN = 128  # slide step          (~8ms  at 16kHz)
N_MELS = 40  # mel bins

# ══════════════════════════════════════════════════════════════
#  STEP 1 — sanitize input
# ══════════════════════════════════════════════════════════════
x = np.array(wave, dtype=np.float32)
# x = x / (np.max(np.abs(x)) + 1e-9)          # normalize to [-1, 1]
print(f"\nWave loaded: {len(x)} samples  |  {len(x) / SR:.2f}s  |  SR={SR}Hz")

# ══════════════════════════════════════════════════════════════
#  STEP 2 — STFT: slice windows, FFT each one
# ══════════════════════════════════════════════════════════════
hann = np.hanning(N_FFT)
n_frames = (len(x) - N_FFT) // HOP_LEN + 1
n_bins = N_FFT // 2 + 1
freqs = np.fft.rfftfreq(N_FFT, 1.0 / SR)

spectrogram = np.zeros((n_bins, n_frames))
for i in range(n_frames):
    frame = x[i * HOP_LEN : i * HOP_LEN + N_FFT] * hann
    spectrogram[:, i] = np.abs(np.fft.rfft(frame)) ** 2  # power

print(f"Spectrogram shape: {spectrogram.shape}  (freq_bins × time_frames)")


# ══════════════════════════════════════════════════════════════
#  STEP 3 — Mel filterbank
# ══════════════════════════════════════════════════════════════
def to_mel(hz):
    return 2595 * np.log10(1 + hz / 700)


def to_hz(mel):
    return 700 * (10 ** (mel / 2595) - 1)


mel_pts = np.linspace(to_mel(0), to_mel(SR / 2), N_MELS + 2)
hz_pts = to_hz(mel_pts)
bin_pts = np.floor((N_FFT + 1) * hz_pts / SR).astype(int)

filterbank = np.zeros((N_MELS, n_bins))
for m in range(1, N_MELS + 1):
    l, c, r = bin_pts[m - 1], bin_pts[m], bin_pts[m + 1]
    filterbank[m - 1, l:c] = (np.arange(l, c) - l) / (c - l + 1e-9)
    filterbank[m - 1, c:r] = (r - np.arange(c, r)) / (r - c + 1e-9)

mel_spec = filterbank @ spectrogram  # (N_MELS, n_frames)
print(f"Mel spec shape:    {mel_spec.shape}  (mel_bins × time_frames)")

# ══════════════════════════════════════════════════════════════
#  STEP 4 — Log compression
# ══════════════════════════════════════════════════════════════
log_mel = np.log(mel_spec + 1e-9)  # (N_MELS, n_frames)
print(f"Log-Mel shape:     {log_mel.shape}  ← this goes into the model")

# ══════════════════════════════════════════════════════════════
#  VISUALIZE
# ══════════════════════════════════════════════════════════════
time_axis = np.arange(n_frames) * HOP_LEN / SR
t_wave = np.arange(len(x)) / SR

fig = plt.figure(figsize=(14, 12))
fig.patch.set_facecolor("#0f1117")
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.6, wspace=0.35)

BG = "#1a1a2e"
GRID = "#2a2a3a"
C = {
    "wave": "#4fc3f7",
    "fft": "#ff8a65",
    "mel": "#a5d6a7",
    "logmel": "#ce93d8",
    "fb": "white",
    "label": "#aaaaaa",
    "title": "#e0e0e0",
}


def ax_style(ax, title, color):
    ax.set_facecolor(BG)
    ax.set_title(title, color=color, fontsize=10, fontweight="bold", pad=7)
    ax.tick_params(colors=C["label"], labelsize=8)
    [s.set_edgecolor(GRID) for s in ax.spines.values()]
    ax.grid(color=GRID, linewidth=0.5)


# ① Waveform
ax1 = fig.add_subplot(gs[0, :])
display_samples = min(len(x), SR * 2)  # show max 2s
ax1.plot(t_wave[:display_samples], x[:display_samples], color=C["wave"], linewidth=0.6)
ax1.set_xlabel("Time (s)", color=C["label"], fontsize=9)
ax1.set_ylabel("Amplitude", color=C["label"], fontsize=9)
ax_style(ax1, "① YOUR WAVEFORM", C["wave"])

# highlight one window in the middle
mid = n_frames // 2
ws, we = mid * HOP_LEN, mid * HOP_LEN + N_FFT
if we < display_samples:
    ax1.axvspan(t_wave[ws], t_wave[we], alpha=0.3, color="yellow")
    ax1.text(
        t_wave[ws],
        0.85,
        f" window\n w{mid}",
        color="yellow",
        fontsize=8,
        transform=ax1.get_xaxis_transform(),
    )

# ② One window zoomed
ax2 = fig.add_subplot(gs[1, 0])
frame_samples = x[ws:we] * hann
ax2.plot(frame_samples, color="yellow", linewidth=1)
ax2.set_xlabel("Sample index", color=C["label"], fontsize=9)
ax2.set_ylabel("Amplitude", color=C["label"], fontsize=9)
ax_style(ax2, f"② WINDOW w{mid}  ({N_FFT / SR * 1000:.0f}ms slice)", "yellow")

# ③ FFT of that window
ax3 = fig.add_subplot(gs[1, 1])
fft_power = np.abs(np.fft.rfft(frame_samples)) ** 2
ax3.plot(freqs, fft_power, color=C["fft"], linewidth=1)
peak_freq = freqs[np.argmax(fft_power)]
ax3.axvline(
    peak_freq,
    color="cyan",
    linestyle="--",
    alpha=0.8,
    label=f"peak: {peak_freq:.0f} Hz",
)
ax3.set_xlabel("Frequency (Hz)", color=C["label"], fontsize=9)
ax3.set_ylabel("Energy", color=C["label"], fontsize=9)
ax3.legend(fontsize=8, facecolor=BG, labelcolor=C["label"], edgecolor=GRID)
ax_style(ax3, f"③ FFT of w{mid}  →  {n_bins} frequency bins", C["fft"])

# ④ Mel filterbank
ax4 = fig.add_subplot(gs[2, 0])
for i in range(N_MELS):
    ax4.plot(
        freqs, filterbank[i], color=plt.cm.rainbow(i / N_MELS), linewidth=1, alpha=0.8
    )
ax4.set_xlabel("Frequency (Hz)", color=C["label"], fontsize=9)
ax4.set_ylabel("Weight", color=C["label"], fontsize=9)
ax_style(ax4, f"④ MEL FILTERBANK  →  {N_MELS} triangular filters", C["mel"])
ax4.text(
    0.98,
    0.93,
    "narrow\n(low freq)",
    transform=ax4.transAxes,
    color="yellow",
    fontsize=7,
    ha="right",
    va="top",
)
ax4.text(
    0.02,
    0.93,
    "wide\n(high freq)",
    transform=ax4.transAxes,
    color="orange",
    fontsize=7,
    ha="left",
    va="top",
)

# ⑤ Spectrogram (before mel)
ax5 = fig.add_subplot(gs[2, 1])
img1 = ax5.imshow(
    np.log(spectrogram + 1e-9),
    aspect="auto",
    origin="lower",
    extent=[time_axis[0], time_axis[-1], 0, SR / 2],
    cmap="magma",
)
ax5.set_xlabel("Time (s)", color=C["label"], fontsize=9)
ax5.set_ylabel("Frequency (Hz)", color=C["label"], fontsize=9)
plt.colorbar(img1, ax=ax5).ax.tick_params(colors=C["label"])
ax_style(ax5, f"⑤ SPECTROGRAM  ({n_bins} bins × {n_frames} frames)", C["fft"])

# ⑥ Log-Mel spectrogram
ax6 = fig.add_subplot(gs[3, :])
img2 = ax6.imshow(
    log_mel,
    aspect="auto",
    origin="lower",
    cmap="viridis",
    extent=[time_axis[0], time_axis[-1], 0.5, N_MELS + 0.5],
)
ax6.set_xlabel("Time (s)", color=C["label"], fontsize=9)
ax6.set_ylabel("Mel bin", color=C["label"], fontsize=9)
cb = plt.colorbar(img2, ax=ax6)
cb.set_label("Log energy", color=C["label"], fontsize=9)
cb.ax.tick_params(colors=C["label"])
ax_style(
    ax6,
    f"⑥ LOG-MEL SPECTROGRAM  ({N_MELS} bins × {n_frames} frames)  ← MODEL INPUT",
    C["logmel"],
)

# mark same window column
ax6.axvline(time_axis[mid], color="yellow", linewidth=1.5, linestyle="--", alpha=0.8)
ax6.text(time_axis[mid] + 0.01, N_MELS - 1, f"w{mid}", color="yellow", fontsize=8)

fig.suptitle(
    "YOUR AUDIO  →  Log-Mel Spectrogram Pipeline",
    color="white",
    fontsize=13,
    fontweight="bold",
    y=0.99,
)

# out = '/mnt/user-data/outputs/your_audio_pipeline.png'
# plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
# print(f"\nSaved → {out}")

# ── print one column of numbers ────────────────────────────────
print(f"\n=== MEL BINS FOR WINDOW w{mid} ===")
for i, (raw, log) in enumerate(zip(mel_spec[:, mid], log_mel[:, mid])):
    bar = "█" * int(max(0, log + 16))
    print(f"  Mel bin {i + 1:2d}: raw={raw:10.2f}  log={log:7.3f}  {bar}")

In [ ]:
def hz_to_mel(f):
    return 2595 * np.log10(1 + f / 700)


def mel_to_hz(m):
    return 700 * (10 ** (m / 2595) - 1)


# examples
hz_to_mel(300)  # → 401.2
hz_to_mel(3000)  # → 2146.1
mel_to_hz(1000)  # → 1000.0  (anchor point)
mel_to_hz(2146)  # → 2999.6  ≈ 3000 Hz